# DS 4002 – Project 3: Leukemia Cell Classification
**Group:** Model Citizens &nbsp;|&nbsp; Shaina Banduri, Neil Parikh, Nishana Dahal

This notebook covers the full analysis pipeline from the MI2 plan:
preprocessing → training (ResNet50 & VGG16) → architecture comparison →
hyperparameter tuning → final test evaluation.

---
### How to run
1. **Before opening Colab:** zip your local `data/C-NMC 2019 (PKG)` folder and upload the zip to Google Drive
2. **Runtime → Change runtime type → T4 GPU**
3. Run **Cell 1** — it mounts Google Drive and unzips the dataset automatically
   - When prompted, click **"Connect to Google Drive"** and allow access
   - Update `ZIP_PATH` in Cell 1 if your zip is not named `C-NMC 2019 (PKG).zip` or is in a subfolder
4. Run all cells top-to-bottom (**Runtime → Run all**)
5. Run the last cell to download output files to your computer

Expected total runtime: ~40–60 min on a T4 GPU.

In [ ]:
# ============================================================
# Cell 1 – Install packages and load dataset from Google Drive
# ============================================================
# Before running:
#   1. Right-click "data/C-NMC 2019 (PKG)" on your PC → compress to zip
#   2. Upload the zip to Google Drive (directly in MyDrive, not a subfolder)
#   3. Run this cell — click "Connect to Google Drive" when prompted
#
# If you placed the zip in a subfolder, update ZIP_PATH below.
# ============================================================
import os, pathlib, zipfile

!pip install -q scikit-learn

# ── Mount Google Drive ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Update this if your zip has a different name or location ─
ZIP_PATH = '/content/drive/MyDrive/C-NMC 2019 (PKG).zip'

# ── Unzip dataset (skip if already extracted) ────────────────
CONTENT = pathlib.Path('/content')
if not any(CONTENT.rglob('C-NMC_training_data')):
    print(f'Unzipping dataset from:\n  {ZIP_PATH}')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content')
    print('Extraction complete.')
else:
    print('Dataset already present — skipping extraction.')

!ls /content/

In [ ]:
# ============================================================
# Cell 2 – Locate data directories (handles different zip layouts)
# ============================================================
import pathlib

CONTENT = pathlib.Path('/content')

# Find training data root recursively
TRAIN_ROOT = next(
    (p for p in CONTENT.rglob('C-NMC_training_data') if p.is_dir()), None
)

# Find prelim test directory and its labels CSV
TEST_DIR = next(
    (p for p in CONTENT.rglob('C-NMC_test_prelim_phase_data') if p.is_dir()), None
)
TEST_CSV = TEST_DIR / 'C-NMC_test_prelim_phase_data_labels.csv' if TEST_DIR else None

assert TRAIN_ROOT is not None, 'Training data not found — re-run Cell 1'
assert TEST_DIR   is not None, 'Test data not found — re-run Cell 1'
assert TEST_CSV   is not None or not TEST_CSV.exists(), 'Test labels CSV not found'

print(f'Training root : {TRAIN_ROOT}')
print(f'Folds         : {[x.name for x in sorted(TRAIN_ROOT.iterdir()) if x.is_dir()]}')
print(f'Test directory: {TEST_DIR}')
print(f'Test CSV      : {TEST_CSV}')

In [ ]:
# ============================================================
# Cell 3 – Imports, configuration, transforms, output dir
# ============================================================
import os, copy
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
)
import matplotlib.pyplot as plt
from PIL import Image

# ── Output directory ─────────────────────────────────────────
OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Device ───────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cpu':
    print('WARNING: No GPU detected. Training will be very slow.\n'
          'Go to Runtime > Change runtime type > GPU.')

# ── Constants (per MI2 analysis plan) ────────────────────────
IMG_SIZE      = 224                          # required by ResNet50/VGG16
IMAGENET_MEAN = [0.485, 0.456, 0.406]        # ImageNet statistics for normalisation
IMAGENET_STD  = [0.229, 0.224, 0.225]

CLASSES       = {'all': 1, 'hem': 0}         # all = leukemia (1), hem = normal (0)
LABEL_NAMES   = ['hem (normal)', 'all (leukemia)']

HEAD_EPOCHS   = 3     # phase-1: train only the new head
MAX_EPOCHS    = 30    # phase-2: max fine-tune epochs (early stopping will usually kick in earlier)
PATIENCE      = 4     # stop if val F1 doesn't improve for this many consecutive epochs
LR_HEAD       = 1e-3  # phase-1 learning rate (head warmup)

# ── Transforms ───────────────────────────────────────────────
# Training: augmentation applied to expand effective dataset size
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Validation / test: only resize + normalise (no augmentation)
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('Config ready.')

In [ ]:
# ============================================================
# Cell 4 – Dataset class and data-loading utilities
# ============================================================

class CnmcDataset(Dataset):
    """PyTorch Dataset for C-NMC 2019 blood smear cell images.

    Parameters
    ----------
    samples   : list of (pathlib.Path, int) — (image_path, label)
    transform : torchvision transform pipeline
    """
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert('RGB')   # BMP → RGB tensor
        if self.transform:
            image = self.transform(image)
        return image, label


def collect_training_samples(train_root, folds, classes):
    """Return [(path, label), ...] for the specified training folds."""
    samples = []
    for fold in folds:
        for cls_name, label in classes.items():
            cls_dir = train_root / fold / cls_name
            for img_path in sorted(cls_dir.glob('*.bmp')):
                samples.append((img_path, label))
    return samples


def load_test_samples(test_dir, labels_csv):
    """Return [(path, label), ...] for the prelim test set.

    The CSV has columns: Patient_ID, new_names, labels
    Images are named by new_names (e.g. 1.bmp, 2.bmp ...).
    labels: 0 = hem/normal, 1 = all/leukemia.
    """
    df = pd.read_csv(labels_csv)
    samples = []
    for _, row in df.iterrows():
        img_path = test_dir / str(row['new_names'])
        samples.append((img_path, int(row['labels'])))
    return samples


def compute_class_weights(samples):
    """Inverse-frequency class weights: weight_c = N / (K * count_c).

    Returns a FloatTensor on DEVICE suitable for CrossEntropyLoss(weight=...).
    """
    counts = Counter(label for _, label in samples)
    n, k   = len(samples), len(counts)
    weights = torch.tensor(
        [n / (k * counts[c]) for c in sorted(counts)],
        dtype=torch.float32,
    ).to(DEVICE)
    return weights


print('Dataset utilities defined.')

In [ ]:
# ============================================================
# Cell 5 – Build fold-based splits and compute class weights
# ============================================================
# Per the MI2 analysis plan:
#   fold_0 + fold_1  →  training
#   fold_2           →  validation (model selection / hyperparameter tuning)
#   C-NMC_test_prelim_phase_data  →  held-out test (touched ONLY in Cell 10)

TRAIN_FOLDS = ['fold_0', 'fold_1']
VAL_FOLDS   = ['fold_2']

train_samples = collect_training_samples(TRAIN_ROOT, TRAIN_FOLDS, CLASSES)
val_samples   = collect_training_samples(TRAIN_ROOT, VAL_FOLDS,   CLASSES)
test_samples  = load_test_samples(TEST_DIR, TEST_CSV)

# Summarise split sizes
for name, samp in [('Train', train_samples), ('Val', val_samples), ('Test', test_samples)]:
    c = Counter(lbl for _, lbl in samp)
    print(f'{name:<6}  hem={c[0]:,}  all={c[1]:,}  total={len(samp):,}')

# Class weights derived from training set only
class_weights = compute_class_weights(train_samples)
print(f'\nClass weights  hem={class_weights[0]:.4f}  all={class_weights[1]:.4f}')

In [ ]:
# ============================================================
# Cell 6 – Model builder
# ============================================================

def build_model(arch: str) -> nn.Module:
    """Load ImageNet-pretrained backbone, replace head with 2-class linear layer.

    Parameters
    ----------
    arch : 'resnet50' or 'vgg16'
    """
    if arch == 'resnet50':
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, 2)
    elif arch == 'vgg16':
        model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, 2)
    else:
        raise ValueError(f'Unknown architecture: {arch}')
    return model.to(DEVICE)


def get_head_params(model, arch):
    """Return only the newly added classification head parameters."""
    if arch == 'resnet50':
        return model.fc.parameters()
    return model.classifier[6].parameters()


print('Model builder defined.')

In [ ]:
# ============================================================
# Cell 7 – Training and validation step functions
# ============================================================

def train_one_epoch(model, loader, criterion, optimizer, scaler):
    """Run one training epoch. Returns weighted F1 on the training mini-batches."""
    model.train()
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        # Mixed-precision forward pass (speeds up training on GPU)
        with torch.amp.autocast(device_type=DEVICE.type,
                                enabled=(DEVICE.type == 'cuda')):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return f1_score(all_labels, all_preds, average='weighted', zero_division=0)


@torch.no_grad()
def evaluate(model, loader):
    """Evaluate model on a DataLoader.

    Returns
    -------
    (weighted_f1, y_true ndarray, y_pred ndarray)
    """
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with torch.amp.autocast(device_type=DEVICE.type,
                                enabled=(DEVICE.type == 'cuda')):
            logits = model(imgs)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    return f1_score(y_true, y_pred, average='weighted', zero_division=0), y_true, y_pred


print('Training utilities defined.')

In [ ]:
# ============================================================
# Cell 8 – Full training pipeline function
# ============================================================
# Two-phase schedule:
#   Phase 1 (HEAD_EPOCHS): freeze backbone, train new head only
#   Phase 2 (up to MAX_EPOCHS): unfreeze all, fine-tune end-to-end
#              with ReduceLROnPlateau and early stopping on val F1

def train_model(arch: str, lr_fine: float, batch_size: int = 32):
    """Train an architecture with the two-phase schedule.

    Parameters
    ----------
    arch       : 'resnet50' or 'vgg16'
    lr_fine    : learning rate for phase-2 full fine-tuning
    batch_size : DataLoader batch size

    Returns
    -------
    dict with keys: arch, lr_fine, best_f1, y_true, y_pred, state_dict, history
    """
    print(f'\n{"="*60}')
    print(f'  Training: {arch.upper()}  |  lr_fine={lr_fine:.0e}  |  batch={batch_size}')
    print(f'{"="*60}')

    # Build data loaders
    pin = DEVICE.type == 'cuda'
    train_loader = DataLoader(
        CnmcDataset(train_samples, transform=train_transform),
        batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=pin,
    )
    val_loader = DataLoader(
        CnmcDataset(val_samples, transform=val_test_transform),
        batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=pin,
    )

    model     = build_model(arch)
    criterion = nn.CrossEntropyLoss(weight=class_weights)   # weighted for class imbalance
    scaler    = torch.amp.GradScaler(device=DEVICE.type,
                                     enabled=(DEVICE.type == 'cuda'))

    # ── Phase 1: head warmup ─────────────────────────────────
    for p in model.parameters():
        p.requires_grad = False
    for p in get_head_params(model, arch):
        p.requires_grad = True

    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD
    )
    print(f'Phase 1 – head warmup ({HEAD_EPOCHS} epochs, lr={LR_HEAD:.0e})')
    for ep in range(HEAD_EPOCHS):
        tr_f1 = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        v_f1, _, _ = evaluate(model, val_loader)
        print(f'  Ep {ep+1:2d}  train_f1={tr_f1:.4f}  val_f1={v_f1:.4f}')

    # ── Phase 2: full fine-tune with early stopping ──────────
    for p in model.parameters():
        p.requires_grad = True

    optimizer = optim.Adam(model.parameters(), lr=lr_fine)
    # Halve LR if val F1 stalls for 2 consecutive epochs
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=2, factor=0.5
    )

    best_f1, best_state  = 0.0, None
    best_yt, best_yp     = None, None
    patience_ctr         = 0
    history              = []

    print(f'Phase 2 – full fine-tune (max {MAX_EPOCHS} ep, patience={PATIENCE}, lr={lr_fine:.0e})')
    for ep in range(MAX_EPOCHS):
        tr_f1 = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        v_f1, y_true, y_pred = evaluate(model, val_loader)
        scheduler.step(v_f1)
        history.append({'epoch': ep + 1, 'train_f1': tr_f1, 'val_f1': v_f1})

        marker = ''
        if v_f1 > best_f1:
            best_f1, best_state = v_f1, copy.deepcopy(model.state_dict())
            best_yt, best_yp    = y_true.copy(), y_pred.copy()
            patience_ctr        = 0
            marker              = '  ← best'
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f'  Ep {ep+1:2d}  train_f1={tr_f1:.4f}  val_f1={v_f1:.4f}  (early stop)')
                break

        print(f'  Ep {ep+1:2d}  train_f1={tr_f1:.4f}  val_f1={v_f1:.4f}{marker}')

    # Save best weights to output/
    run_tag = f'{arch}_{lr_fine:.0e}'.replace('-', 'n')
    torch.save(best_state, OUTPUT_DIR / f'{run_tag}_best.pth')
    print(f'\n  >> Best val F1: {best_f1:.4f}  — saved {run_tag}_best.pth')

    return {
        'arch': arch, 'lr_fine': lr_fine, 'batch_size': batch_size,
        'best_f1': best_f1, 'y_true': best_yt, 'y_pred': best_yp,
        'state_dict': best_state, 'history': history,
    }


print('train_model() defined.')

In [ ]:
# ============================================================
# Cell 9 – Step 1: Architecture Comparison
# Train ResNet50 and VGG16 with the base fine-tune LR (1e-4).
# (~15-25 min per model on T4 GPU)
# ============================================================

BASE_LR_FINE = 1e-4
BASE_BS      = 32

arch_results = {}
for arch in ['resnet50', 'vgg16']:
    arch_results[arch] = train_model(arch, lr_fine=BASE_LR_FINE, batch_size=BASE_BS)

print()
print('=' * 50)
print('Architecture Comparison Summary')
print('=' * 50)
print(f'{"Architecture":<12}  {"Val F1 (weighted)":>20}  {"Target met (≥0.90)":>20}')
print('-' * 56)
for arch, r in arch_results.items():
    met = 'YES' if r['best_f1'] >= 0.90 else 'no'
    print(f'{arch:<12}  {r["best_f1"]:>20.4f}  {met:>20}')

# Select the architecture with the highest validation F1
best_arch = max(arch_results, key=lambda a: arch_results[a]['best_f1'])
print(f'\nSelected architecture: {best_arch.upper()}  (val F1 = {arch_results[best_arch]["best_f1"]:.4f})')

In [ ]:
# ============================================================
# Cell 10 – Step 2: Hyperparameter Tuning (LR sweep on winner)
# Test a lower fine-tune LR on the winning architecture to see
# if a smaller LR extracts more performance.
# ============================================================

ALT_LR = 5e-5   # alternative fine-tune LR to compare against BASE_LR_FINE (1e-4)

print(f'Running LR sweep on {best_arch.upper()}: comparing LR={BASE_LR_FINE:.0e} vs LR={ALT_LR:.0e}')

# Train winner with the alternative LR (base result already in arch_results)
alt_result = train_model(best_arch, lr_fine=ALT_LR, batch_size=BASE_BS)

# Collect both LR results for comparison
hp_results = {
    BASE_LR_FINE: arch_results[best_arch],   # reused from Cell 9 — no redundant training
    ALT_LR:       alt_result,
}

print()
print('=' * 50)
print(f'Hyperparameter Tuning — {best_arch.upper()}')
print('=' * 50)
print(f'{"LR (fine-tune)":<18}  {"Val F1 (weighted)":>20}')
print('-' * 42)
for lr, r in hp_results.items():
    print(f'{lr:<18.0e}  {r["best_f1"]:>20.4f}')

# Best LR = whichever produced the highest val F1
best_lr     = max(hp_results, key=lambda lr: hp_results[lr]['best_f1'])
best_result = hp_results[best_lr]

print(f'\nBest config: {best_arch.upper()}  LR_fine={best_lr:.0e}')
print(f'Best val F1: {best_result["best_f1"]:.4f}')

In [ ]:
# ============================================================
# Cell 11 – Step 3: Final Evaluation on Held-Out Test Set
# This cell is run ONCE with the best model.
# The test set is NEVER used for model selection or tuning.
# ============================================================

print(f'Evaluating best model on held-out test set...')
print(f'  Architecture : {best_arch.upper()}')
print(f'  LR fine-tune : {best_lr:.0e}')
print()

# Reload best model weights
best_model = build_model(best_arch)
best_model.load_state_dict(best_result['state_dict'])

test_loader = DataLoader(
    CnmcDataset(test_samples, transform=val_test_transform),
    batch_size=BASE_BS, shuffle=False, num_workers=2, pin_memory=(DEVICE.type == 'cuda'),
)

test_f1, test_y_true, test_y_pred = evaluate(best_model, test_loader)

test_precision = precision_score(test_y_true, test_y_pred, average='weighted', zero_division=0)
test_recall    = recall_score(test_y_true, test_y_pred, average='weighted', zero_division=0)

print(f'Test weighted F1  : {test_f1:.4f}   (target ≥ 0.90 — {"MET" if test_f1 >= 0.90 else "NOT MET"})')
print(f'Test precision    : {test_precision:.4f}')
print(f'Test recall       : {test_recall:.4f}')
print()
print('=== Full Classification Report ===')
print(classification_report(test_y_true, test_y_pred, target_names=LABEL_NAMES, zero_division=0))

# Save report to file
report_text = (
    f'Best model: {best_arch.upper()}  |  LR_fine={best_lr:.0e}  |  Batch={BASE_BS}\n'
    f'Val weighted F1 : {best_result["best_f1"]:.4f}\n'
    f'Test weighted F1: {test_f1:.4f}\n'
    f'Test precision  : {test_precision:.4f}\n'
    f'Test recall     : {test_recall:.4f}\n\n'
    + classification_report(test_y_true, test_y_pred, target_names=LABEL_NAMES, zero_division=0)
)
with open(OUTPUT_DIR / 'classification_report.txt', 'w') as f:
    f.write(report_text)
print('Saved: classification_report.txt')

In [ ]:
# ============================================================
# Cell 12 – Generate and save all output figures
# ============================================================

# ── Figure 1: Architecture comparison (val F1) ──────────────
fig, ax = plt.subplots(figsize=(5, 4))
arch_names = list(arch_results.keys())
arch_f1s   = [arch_results[a]['best_f1'] for a in arch_names]
bars = ax.bar(arch_names, arch_f1s, color=['#2196F3', '#FF9800'],
              edgecolor='black', linewidth=0.7)
ax.axhline(0.90, color='red', linestyle='--', linewidth=1.2, label='0.90 target')
ax.set_ylim(max(0.78, min(arch_f1s) - 0.05), 1.0)
ax.set_ylabel('Validation Weighted F1')
ax.set_title('ResNet50 vs VGG16 — Validation F1')
ax.legend(fontsize=9)
for bar, v in zip(bars, arch_f1s):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.003,
            f'{v:.4f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'model_comparison.png', dpi=150)
plt.show()
print('Saved: model_comparison.png')

# ── Figure 2: LR tuning comparison ──────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
hp_lr_labels = [f'{lr:.0e}' for lr in hp_results]
hp_f1s       = [hp_results[lr]['best_f1'] for lr in hp_results]
bars = ax.bar(hp_lr_labels, hp_f1s, color='#4CAF50',
              edgecolor='black', linewidth=0.7)
ax.axhline(0.90, color='red', linestyle='--', linewidth=1.2, label='0.90 target')
ax.set_ylim(max(0.78, min(hp_f1s) - 0.05), 1.0)
ax.set_xlabel(f'Fine-tune LR ({best_arch.upper()})')
ax.set_ylabel('Validation Weighted F1')
ax.set_title(f'LR Sweep — {best_arch.upper()}')
ax.legend(fontsize=9)
for bar, v in zip(bars, hp_f1s):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.003,
            f'{v:.4f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'hp_tuning.png', dpi=150)
plt.show()
print('Saved: hp_tuning.png')

# ── Figure 3: Confusion matrix (test set) ───────────────────
cm = confusion_matrix(test_y_true, test_y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(LABEL_NAMES, rotation=15, ha='right', fontsize=9)
ax.set_yticklabels(LABEL_NAMES, fontsize=9)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True', fontsize=11)
ax.set_title(f'Confusion Matrix — {best_arch.upper()} (test set)', fontsize=10)
thresh = cm.max() / 2
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black', fontsize=13)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=150)
plt.show()
print('Saved: confusion_matrix.png')

# ── Figure 4: Training curves (best model, fine-tune phase) ─
history  = best_result['history']
epochs   = [h['epoch']    for h in history]
tr_f1s   = [h['train_f1'] for h in history]
vl_f1s   = [h['val_f1']   for h in history]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs, tr_f1s, marker='o', markersize=4, label='Train F1')
ax.plot(epochs, vl_f1s, marker='s', markersize=4, label='Val F1')
ax.axhline(0.90, color='red', linestyle='--', linewidth=1, label='0.90 target')
ax.set_xlabel('Epoch (fine-tune phase)')
ax.set_ylabel('Weighted F1')
ax.set_title(f'Training Curves — {best_arch.upper()} (LR={best_lr:.0e})')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150)
plt.show()
print('Saved: training_curves.png')

print(f'\nAll figures saved to {OUTPUT_DIR}')

In [ ]:
# ============================================================
# Cell 13 – Download all output files to your local machine
# ============================================================
# Save these files into your local output/ folder.

from google.colab import files

print('Downloading output files...')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = str(OUTPUT_DIR / fname)
    print(f'  {fname}')
    files.download(fpath)
print('Done.')